# Generating Graphs

In [261]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from itertools import cycle
import os
import re

# Load Data

In [ ]:
output_dir = "../graphs/ablation_bertimbau"
os.makedirs(output_dir, exist_ok=True)

filepaths = ["../data/eval/results_eval_mteb_v1_11_datasets_w_mteb.csv", "../data/eval/results_eval_mteb_default.csv", "../data/eval/results_eval_mteb_secondary.csv"]

filepath_finetuned_matryoshka = "../data/eval/results_eval_mteb_v1_11_datasets_w_mteb.csv"
filepath_baseline = "../data/eval/results_eval_mteb_default.csv"
model_names = []#['BERTimbau-large-matryoshka-sts-pt', 'BERTimbau-large-sts-pt'] 
model_names_to_remove = [] #['BERTimbau-large-sts-pt', 'e5-large-sts-pt', 'bert-base-portuguese-cased', 'mmBERT-base']

results = [pd.read_csv(filepath) for filepath in filepaths]

# Pre-processing

In [263]:
task_type_map = {
    "Assin2STS": "STS",
    "SICK-BR-STS": "STS",
    "STSBenchmarkMultilingualSTS": "STS",
    'MassiveIntentClassification': "Classification",
    'MultiHateClassification': "Classification",
    'BrazilianToxicTweetsClassification': "Classification",
    'HateSpeechPortugueseClassification': "Classification",
    'TweetSentimentClassification': "Classification",
    'MultiLongDocReranking': "Reranking",
    'WikipediaRerankingMultilingual': "Reranking",
    'XGlueWPRReranking': "Reranking",
    'WebFAQRetrieval': "Retrieval",
    'MultiLongDocRetrieval': "Retrieval",
    'WikipediaRetrievalMultilingual': "Retrieval",
}

max_emb_dim_map = {
    'paraphrase-multilingual-MiniLM-L12-v2': 384,
    'BERTimbau': 1024,
    'Qwen3': 1024,
    'bert-base-portuguese-cased': 1024,
    'mmBERT': 768,
    'e5': 1024,
    'ModBERTBr': 768,
    'NeoBERTugues': 768,
    'Qwen3-Embedding-0.6B-sts-pt': 1024
}

In [264]:
def map_model_name(model_raw_name: str, mapping: dict) -> int:
    matches = [
        value
        for key, value in mapping.items()
        if key.lower() in model_raw_name.lower()
    ]

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found for '{model_raw_name}': "
            f"{[k for k in mapping if k.lower() in model_raw_name.lower()]}"
        )
    if len(matches) == 0:
        return None  # ou raise ValueError se quiser exigir match obrigatório

    return matches[0]

In [265]:
results_all = pd.concat(results, ignore_index=True)
results_all['task_type'] = results_all['task_name'].map(task_type_map)
# results_all.dropna(subset=['task_type'], inplace=True)
results_all['model_raw_name'] = results_all['model_name'].apply(lambda x: x.split("/")[-1])
results_all["max_emb_size"] = results_all["model_raw_name"].apply(
    lambda x: map_model_name(x, max_emb_dim_map)
)
results_all['embedding_dim_full'] = results_all['embedding_dim'].combine_first(results_all['max_emb_size']).astype(int)

if model_names:
    results_all = results_all.query("model_raw_name.isin(@model_names)")

if model_names_to_remove:
    results_all = results_all.query("~model_raw_name.isin(@model_names_to_remove)")

results_all.head()

,model_name,embedding_dim,task_name,main_score,task_type,model_raw_name,max_emb_size,embedding_dim_full
70,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,Assin2STS,0.829265,STS,BERTimbau-large-matryoshka-sts-pt,1024,1024
71,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,SICK-BR-STS,0.906358,STS,BERTimbau-large-matryoshka-sts-pt,1024,1024
72,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,0.847668,STS,BERTimbau-large-matryoshka-sts-pt,1024,1024
73,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,MassiveIntentClassification,0.634997,Classification,BERTimbau-large-matryoshka-sts-pt,1024,1024
74,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,MultiHateClassification,0.633100,Classification,BERTimbau-large-matryoshka-sts-pt,1024,1024


In [266]:
# cond_e5 = results_all['model_raw_name'].str.contains('e5')
# cond_bertimbau = results_all['model_raw_name'].str.contains('BERTimbau')
# cond_qwen = results_all['model_raw_name'].str.contains('Qwen')
# cond_mmbert_embed = results_all['model_raw_name'].str.contains('mmbert-embed')
# results_all = results_all.loc[cond_e5 | cond_bertimbau | cond_qwen | cond_mmbert_embed]
results_all.head()

,model_name,embedding_dim,task_name,main_score,task_type,model_raw_name,max_emb_size,embedding_dim_full
70,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,Assin2STS,0.829265,STS,BERTimbau-large-matryoshka-sts-pt,1024,1024
71,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,SICK-BR-STS,0.906358,STS,BERTimbau-large-matryoshka-sts-pt,1024,1024
72,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,0.847668,STS,BERTimbau-large-matryoshka-sts-pt,1024,1024
73,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,MassiveIntentClassification,0.634997,Classification,BERTimbau-large-matryoshka-sts-pt,1024,1024
74,iara-project/BERTimbau-large-matryoshka-sts-pt,NaN,MultiHateClassification,0.633100,Classification,BERTimbau-large-matryoshka-sts-pt,1024,1024


# Summarized Metrics

In [267]:
results_all.model_raw_name.unique()

<ArrowStringArray>
['BERTimbau-large-matryoshka-sts-pt', 'BERTimbau-large-sts-pt']
Length: 2, dtype: str

In [268]:
baseline_model = 'Qwen3-Embedding-0.6B'
compare_model = 'e5-large-matryoshka-sts-pt'

In [269]:
grouped_results = results_all.query("model_raw_name.isin([@baseline_model, @compare_model])") \
                    .groupby(["model_raw_name", "task_type"])['main_score'] \
                    .mean() \
                    .reset_index()
grouped_results

,model_raw_name,task_type,main_score


In [270]:
def percent_diff_by_task_type(grouped_results, baseline_model, compare_model):
    wide = (
        grouped_results
        .pivot(index="task_type", columns="model_raw_name", values="main_score")
        .reindex(columns=[baseline_model, compare_model])
    )

    out = pd.DataFrame(index=wide.index)
    out["baseline"] = wide[baseline_model]
    out["compare"]  = wide[compare_model]
    out["abs_diff"] = out["compare"] - out["baseline"]

    # avoid division-by-zero issues
    out["pct_diff_vs_baseline"] = np.where(
        out["baseline"].ne(0),
        (out["abs_diff"] / out["baseline"]) * 100.0,
        np.nan
    )

    return out.reset_index().sort_values("task_type")

# usage
diff_table = percent_diff_by_task_type(grouped_results, baseline_model, compare_model)
diff_table

,task_type,baseline,compare,abs_diff,pct_diff_vs_baseline


In [271]:
df_results_ratio = pd.DataFrame()

results_all["num_dims"] = results_all["embedding_dim"].fillna(results_all["embedding_dim_full"])
results_all["num_dims"] = pd.to_numeric(results_all["num_dims"], errors="coerce")
df_ratio = results_all.query("max_emb_size > 768").copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce").astype(int)
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])

df_ratio = (
    df_ratio
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)

task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = df_tt[df_tt["num_dims"] == df_tt["max_dim"]][
        ["model_raw_name", "main_score"]
    ].rename(columns={"main_score": "max_score"})

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]

    df_results_ratio = pd.concat([df_results_ratio, df_tt], axis=0, ignore_index=True)
df_results_ratio

,task_type,model_raw_name,num_dims,main_score,max_dim,max_score,ratio
0,Classification,BERTimbau-large-matryoshka-sts-pt,64,0.476538,1024,0.518151,0.919690
1,Classification,BERTimbau-large-matryoshka-sts-pt,128,0.494454,1024,0.518151,0.954267
2,Classification,BERTimbau-large-matryoshka-sts-pt,256,0.503793,1024,0.518151,0.972290
3,Classification,BERTimbau-large-matryoshka-sts-pt,512,0.512998,1024,0.518151,0.990055
4,Classification,BERTimbau-large-matryoshka-sts-pt,1024,0.518151,1024,0.518151,1.000000
5,Classification,BERTimbau-large-sts-pt,64,0.457486,1024,0.511298,0.894754
6,Classification,BERTimbau-large-sts-pt,128,0.482997,1024,0.511298,0.944648
7,Classification,BERTimbau-large-sts-pt,256,0.503300,1024,0.511298,0.984358
8,Classification,BERTimbau-large-sts-pt,512,0.511179,1024,0.511298,0.999767
9,Classification,BERTimbau-large-sts-pt,1024,0.511298,1024,0.511298,1.000000


In [272]:
print(compare_model)
df_compare_ratio = df_results_ratio.query("model_raw_name == @compare_model & num_dims == 64")
print(f"Média Ratio: {df_compare_ratio['ratio'].mean()}")
df_compare_ratio

e5-large-matryoshka-sts-pt
Média Ratio: nan


,task_type,model_raw_name,num_dims,main_score,max_dim,max_score,ratio


In [273]:
print(baseline_model)
df_compare_ratio = df_results_ratio.query("model_raw_name == @baseline_model & num_dims == 64")
print(f"Média Ratio: {df_compare_ratio['ratio'].mean()}")
df_compare_ratio

Qwen3-Embedding-0.6B
Média Ratio: nan


,task_type,model_raw_name,num_dims,main_score,max_dim,max_score,ratio


# Export Graphs

In [274]:
import os
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


# -------------------------
# Utils
# -------------------------
def sanitize_filename(name):
    name = str(name)
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.lower()


def apply_y_axis_formatting(ax, series, step=0.03, margin=0.01):
    y_min = float(series.min())
    y_max = float(series.max())

    y_bottom = (np.floor(y_min / step) * step) - margin
    y_top    = (np.floor(y_max / step) * step) + step + margin

    ax.set_ylim(y_bottom, y_top)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.tick_params(axis="y", labelsize=14)
    ax.tick_params(axis="x", labelsize=14)


def apply_log2_xaxis(ax, x_order, left_pad=1.08, right_pad=1.18):
    """
    Match the reference spacing (8,16,32,...) using log base 2.
    Adds a bit of right padding so end-labels have room.
    """
    x_order = sorted(pd.unique(pd.Series(x_order).dropna()))
    ax.set_xscale("log", base=2)
    ax.set_xticks(x_order)

    x_set = set([float(x) for x in x_order])

    def _fmt(x, pos):
        if float(x) in x_set:
            if abs(x - round(x)) < 1e-9:
                return str(int(round(x)))
            return f"{x:g}"
        return ""

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt))

    # Padding for labels on the right
    x_min = float(min(x_order))
    x_max = float(max(x_order))
    ax.set_xlim(x_min / left_pad, x_max * right_pad)


def make_model_style_map(model_names):
    model_names = list(model_names)

    if len(model_names) <= 10:
        colors = sns.color_palette("tab10", n_colors=len(model_names))
    else:
        colors = sns.color_palette("husl", n_colors=len(model_names))

    markers = ["o", "s", "^", "v", "D", "P", "X", "<", ">", "h", "p", "8"]
    dash_patterns = [
        None,          # solid
        (6, 2),        # dashed
        (2, 2),        # densely dashed
        (6, 2, 1, 2),  # dash-dot-ish
        (1, 1),        # dotted
    ]

    style_map = {}
    for i, m in enumerate(model_names):
        style_map[m] = {
            "color": colors[i],
            "marker": markers[i % len(markers)],
            "dashes": dash_patterns[i % len(dash_patterns)],
        }
    return style_map


def build_custom_lineplot(
    ax,
    data,
    style_map,
    highlight_models=None,
    highlight_kwargs=None,
    other_kwargs=None,
):
    """
    Emphasis strategy:
      - non-highlight models: lower alpha, thinner lines, smaller markers
      - highlighted models: thicker lines, bigger markers, higher zorder (drawn on top)
    Returns:
      - end_points: list of dicts with x,y,text,color for labels (ONLY highlight models by default)
    """
    highlight_models = set(highlight_models or [])

    highlight_kwargs = highlight_kwargs or dict(
        linewidth=3.4,
        markersize=10.0,
        alpha=1.0,
        zorder=6,
        markeredgewidth=1.0,
        markeredgecolor="white",
    )
    other_kwargs = other_kwargs or dict(
        linewidth=1.8,
        markersize=7.0,
        alpha=1.0,
        zorder=2,
        markeredgewidth=0.0,
    )

    end_points = []

    # Draw others first, highlights last (so highlights sit on top)
    models_in_data = list(pd.unique(data["model_raw_name"]))
    draw_order = [m for m in models_in_data if m not in highlight_models] + \
                 [m for m in models_in_data if m in highlight_models]

    for model in draw_order:
        df_m = data[data["model_raw_name"] == model].sort_values("num_dims")
        if df_m.empty:
            continue

        st = style_map.get(model, {})
        is_highlight = model in highlight_models
        kw = highlight_kwargs if is_highlight else other_kwargs

        (line,) = ax.plot(
            df_m["num_dims"].to_numpy(),
            df_m["main_score"].to_numpy(),
            label=model,
            color=st.get("color", None),
            marker=st.get("marker", "o"),
            linewidth=kw["linewidth"],
            markersize=kw["markersize"],
            alpha=kw["alpha"],
            zorder=kw["zorder"],
            markeredgewidth=kw.get("markeredgewidth", 0.0),
            markeredgecolor=kw.get("markeredgecolor", None),
        )
        if st.get("dashes") is not None:
            line.set_dashes(st["dashes"])

        # Collect last-point info for highlighted labels
        if is_highlight:
            last = df_m.iloc[-1]
            end_points.append(
                dict(
                    model=model,
                    x=float(last["num_dims"]),
                    y=float(last["main_score"]),
                    color=st.get("color", "black"),
                )
            )

    return end_points


def annotate_end_labels_with_vertical_spacing(
    ax,
    end_points,
    x_mult=1.06,
    min_sep_frac=0.03,
    clamp=True,
):
    """
    Places labels near the last point of each line, but "repels" them vertically
    so they don't overlap (useful when two end labels are too close).

    - min_sep_frac: fraction of the y-range to enforce as minimum vertical separation
    - x_mult: puts text slightly to the right (works well with log-x)
    """
    if not end_points:
        return

    y0, y1 = ax.get_ylim()
    y_span = (y1 - y0) if (y1 - y0) != 0 else 1.0
    min_sep = y_span * float(min_sep_frac)

    # Sort by y and push labels apart
    pts = sorted(end_points, key=lambda d: d["y"])
    y_adj = []
    for i, d in enumerate(pts):
        y_target = d["y"]
        if i == 0:
            y_new = y_target
        else:
            y_new = max(y_target, y_adj[-1] + min_sep)
        y_adj.append(y_new)

    # If we pushed beyond the top, shift everything down
    if clamp:
        overflow = y_adj[-1] - y1
        if overflow > 0:
            y_adj = [yy - overflow for yy in y_adj]
        # If now below bottom, shift up
        underflow = y0 - y_adj[0]
        if underflow > 0:
            y_adj = [yy + underflow for yy in y_adj]

    # Draw annotations (with a light connector line)
    for d, yy in zip(pts, y_adj):
        x_text = d["x"] * float(x_mult)
        ax.annotate(
            d["model"],
            xy=(d["x"], d["y"]),
            xytext=(x_text, yy),
            textcoords="data",
            fontsize=10.5,
            fontweight="bold",
            color=d["color"],
            va="center",
            ha="left",
            zorder=10,
            arrowprops=dict(arrowstyle="-", lw=0.8, color=d["color"], alpha=0.7),
        )


def fix_legend_inside_bottom_right(ax, highlight_models=None):
    """
    Keep legend inside bottom-right. Reorder to show highlighted models first.
    """
    highlight_models = set(highlight_models or [])
    handles, labels = ax.get_legend_handles_labels()

    idx_high = [i for i, lab in enumerate(labels) if lab in highlight_models]
    idx_rest = [i for i, lab in enumerate(labels) if lab not in highlight_models]
    new_idx = idx_high + idx_rest

    handles = [handles[i] for i in new_idx]
    labels  = [labels[i] for i in new_idx]

    leg = ax.legend(
        handles,
        labels,
        loc="lower right",
        bbox_to_anchor=(0.98, 0.06),
        frameon=True,
        framealpha=0.92,
        fontsize=10.5,
        borderaxespad=0.0,
        handlelength=3.0,
        labelspacing=0.35,
    )
    leg.get_frame().set_linewidth(0.8)


# -------------------------
# Theme
# -------------------------
sns.set_theme(style="whitegrid", context="talk", font_scale=1.05)


# =========================
# 1) Preparação dos dados
# =========================
df_plot = results_all.copy()

df_plot["num_dims"] = df_plot["embedding_dim"].fillna(df_plot["embedding_dim_full"])
df_plot["num_dims"] = pd.to_numeric(df_plot["num_dims"], errors="coerce")
df_plot["main_score"] = pd.to_numeric(df_plot["main_score"], errors="coerce")

df_plot = df_plot.dropna(subset=["num_dims", "main_score", "model_raw_name"])
df_plot["model_raw_name"] = df_plot["model_raw_name"].astype(str)

model_order = sorted(df_plot["model_raw_name"].unique())
MODEL_STYLE = make_model_style_map(model_order)

# ✅ Models to emphasize (and label at the end)
HIGHLIGHT_MODELS = {
    "BERTimbau-large-matryoshka-sts-pt",
    "e5-large-matryoshka-sts-pt",
}


# =========================================================
# 2) Gráficos por task_name
# =========================================================
task_names = sorted(df_plot["task_name"].dropna().unique())

for task in task_names:
    df_task = (
        df_plot[df_plot["task_name"] == task]
        .groupby(["task_name", "model_raw_name", "num_dims"], as_index=False)["main_score"]
        .mean()
    )

    x_order = sorted(df_task["num_dims"].unique())

    fig, ax = plt.subplots(figsize=(14, 7))
    end_points = build_custom_lineplot(
        ax,
        df_task,
        MODEL_STYLE,
        highlight_models=HIGHLIGHT_MODELS,
    )

    ax.set_title(f"Score per Dimension - Task: {task}", fontsize=20, fontweight="bold", pad=10)
    ax.set_xlabel("Embedding Dimension", fontsize=16)
    ax.set_ylabel("Score", fontsize=16)

    apply_log2_xaxis(ax, x_order)
    apply_y_axis_formatting(ax, df_task["main_score"], step=0.03, margin=0.01)

    # ✅ Add vertical spacing between end labels when too close
    annotate_end_labels_with_vertical_spacing(
        ax,
        end_points,
        x_mult=1.06,
        min_sep_frac=0.035,  # increase if you still see overlap
        clamp=True,
    )

    fix_legend_inside_bottom_right(ax, highlight_models=HIGHLIGHT_MODELS)
    plt.tight_layout()

    filename = f"task_{sanitize_filename(task)}.png"
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close(fig)


# =========================================================
# 3) Gráficos por task_type (média)
# =========================================================
df_task_type = (
    df_plot
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)

task_types = sorted(df_task_type["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_task_type[df_task_type["task_type"] == ttype].copy()
    x_order = sorted(df_tt["num_dims"].unique())

    fig, ax = plt.subplots(figsize=(14, 7))
    end_points = build_custom_lineplot(
        ax,
        df_tt,
        MODEL_STYLE,
        highlight_models=HIGHLIGHT_MODELS,
    )

    ax.set_title(f"Score Per Dimension - Task Type: {ttype}", fontsize=20, fontweight="bold", pad=10)
    ax.set_xlabel("Embedding Dimension", fontsize=16)
    ax.set_ylabel("Score", fontsize=16)

    apply_log2_xaxis(ax, x_order)
    apply_y_axis_formatting(ax, df_tt["main_score"], step=0.03, margin=0.01)

    annotate_end_labels_with_vertical_spacing(
        ax,
        end_points,
        x_mult=1.06,
        min_sep_frac=0.035,
        clamp=True,
    )

    fix_legend_inside_bottom_right(ax, highlight_models=HIGHLIGHT_MODELS)
    plt.tight_layout()

    filename = f"task_type_{sanitize_filename(ttype)}.png"
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close(fig)

In [275]:
# =========================================================
# 5) Ratio vs full embedding por task_type
# =========================================================

map_lim_task = {
    'Classification': 0.8,
    'Reranking': 0.85,
    'Retrieval': 0.35,
    'STS': 0.9
}

df_ratio = df_plot.query("max_emb_size > 768").copy()

df_ratio["num_dims"] = pd.to_numeric(df_ratio["num_dims"], errors="coerce").astype(int)
df_ratio["main_score"] = pd.to_numeric(df_ratio["main_score"], errors="coerce")

df_ratio = df_ratio.dropna(subset=["task_type", "model_raw_name", "num_dims", "main_score"])

df_ratio = (
    df_ratio
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)


def get_hue_order(models):
    """
    Retorna a lista de modelos ordenada: primeiro os sem matryoshka,
    depois os com matryoshka — agrupando visualmente as barras.
    """
    standard    = sorted([m for m in models if "matryoshka" not in m.lower()])
    matryoshka  = sorted([m for m in models if "matryoshka" in m.lower()])
    return standard + matryoshka


def apply_bar_hatch(ax, hue_order):
    """
    Aplica hachura nos modelos com 'matryoshka' e preenchimento
    sólido nos demais. Atualiza a legenda em dois blocos separados:
    um para os modelos (cores) e outro para o tipo de preenchimento.
    """
    handles, labels = ax.get_legend_handles_labels()

    for container, label in zip(ax.containers, labels):
        hatch = "//" if "matryoshka" in label.lower() else ""
        for bar in container:
            bar.set_hatch(hatch)

    # Espelha o hatch nos patches da legenda dos modelos
    for handle, label in zip(handles, labels):
        hatch = "//" if "matryoshka" in label.lower() else ""
        handle.set_hatch(hatch)

    # --- Bloco 1: legenda dos modelos (cores + hatch) ---
    leg1 = ax.legend(
        handles, labels,
        title="Modelo",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
        title_fontsize=10,
    )
    ax.add_artist(leg1)

    # --- Bloco 2: legenda do tipo de preenchimento ---
    solid_patch   = mpatches.Patch(facecolor="white", edgecolor="black", hatch="",   label="Matryoshka")
    hatched_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="//", label="Standard")

    ax.legend(
        handles=[solid_patch, hatched_patch],
        title="Loss",
        bbox_to_anchor=(1.02, 0.0),
        loc="lower left",
        fontsize=9,
        title_fontsize=10,
    )


task_types = sorted(df_ratio["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_ratio[df_ratio["task_type"] == ttype].copy()

    max_dims = (
        df_tt.groupby("model_raw_name")["num_dims"]
        .max()
        .rename("max_dim")
        .reset_index()
    )
    df_tt = df_tt.merge(max_dims, on="model_raw_name", how="left")

    df_max = df_tt[df_tt["num_dims"] == df_tt["max_dim"]][
        ["model_raw_name", "main_score"]
    ].rename(columns={"main_score": "max_score"})

    df_tt = df_tt.merge(df_max, on="model_raw_name", how="left")
    df_tt["ratio"] = df_tt["main_score"] / df_tt["max_score"]

    # Ordena hue: Standard primeiro, Matryoshka depois
    hue_order = get_hue_order(df_tt["model_raw_name"].unique())

    fig, ax = plt.subplots(figsize=(12, 6))

    sns.barplot(
        data=df_tt,
        x="num_dims",
        y="ratio",
        hue="model_raw_name",
        hue_order=hue_order,
        ax=ax,
    )

    ax.set_title(f"Ratio vs Embedding Completo - Task Type: {ttype}")
    ax.set_xlabel("Embedding dimension")
    ax.set_ylabel("Ratio of maximum performance")
    ax.set_ylim(map_lim_task[ttype], 1)

    apply_bar_hatch(ax, hue_order)

    plt.tight_layout(rect=[0, 0, 0.9, 1])

    filename = f"ratio_task_type_{sanitize_filename(ttype)}.png"
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close(fig);